In [1]:
!pip install -q pythainlp rank-bm25 sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.8/19.8 MB 72.2 MB/s eta 0:00:00:00:0100:01


In [2]:
import os
import pandas as pd
import numpy as np
from pathlib import Path
import re
import time
import requests
import json
import torch
from pythainlp.tokenize import word_tokenize
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from kaggle_secrets import UserSecretsClient

# --- 1. CONFIGURATION (ใช้ Path ตามที่คุณระบุมาเป๊ะๆ) ---
# โฟลเดอร์หลักที่มีไฟล์ questions.csv
DATA_DIR = "/kaggle/input/datasets/atlz2z/fah-mai-rag-challenge-level-1/data"
# โฟลเดอร์ Knowledge Base ที่มีโฟลเดอร์ย่อย 3 อัน
KB_DIR = os.path.join(DATA_DIR, "knowledge_base")

# --- 2. LOAD & CHUNK DATA ---
def load_kb():
    kb_path = Path(KB_DIR)
    documents = []
    
    # ระบุโฟลเดอร์ย่อยทั้ง 3 ตามที่คุณแจ้งมา
    sub_folders = ['policies', 'products', 'store_info']
    
    for folder in sub_folders:
        current_folder_path = kb_path / folder
        if not current_folder_path.exists():
            print(f"⚠️ Warning: ไม่พบโฟลเดอร์ {current_folder_path}")
            continue
            
        # ค้นหาไฟล์ .md ทั้งหมดในโฟลเดอร์ย่อยนั้นๆ
        md_files = list(current_folder_path.glob("*.md"))
        print(f"📁 กำลังโหลดจาก {folder}: พบ {len(md_files)} ไฟล์")
        
        for fp in md_files:
            try:
                text = fp.read_text(encoding="utf-8")
                # เก็บ source เป็น 'ชื่อโฟลเดอร์/ชื่อไฟล์' เช่น 'products/Watch_S3.md'
                documents.append({
                    "source": f"{folder}/{fp.name}",
                    "content": text
                })
            except Exception as e:
                print(f"❌ Error reading {fp.name}: {e}")
                
    return documents

# เริ่มการโหลด
docs = load_kb()

if len(docs) == 0:
    print("❌ ไม่สามารถโหลดเอกสารได้เลย! กรุณาตรวจสอบ Path อีกครั้ง")
    # ลองรันคำสั่งนี้เพื่อเช็คว่าในโฟลเดอร์นั้นมีอะไรบ้าง
    print("--- รายชื่อไฟล์ที่มีในระบบ ---")
    !ls -R /kaggle/input/datasets/atlz2z/fah-mai-rag-challenge-level-1/data/knowledge_base
else:
    print(f"✅ โหลดสำเร็จ! รวมทั้งหมด {len(docs)} เอกสาร")

# --- ฟังก์ชัน Chunking (เหมือนเดิมแต่ปรับปรุงความเสถียร) ---
def get_chunks(documents, size=700, overlap=150):
    chunks = []
    for d in documents:
        content = d['content']
        source = d['source']
        if len(content) <= size:
            chunks.append({"text": content, "source": source})
        else:
            start = 0
            while start < len(content):
                end = start + size
                chunks.append({"text": content[start:end], "source": source})
                start += size - overlap
    return chunks

all_chunks = get_chunks(docs)
print(f"✅ สร้างได้ทั้งหมด {len(all_chunks)} chunks.")

# --- 3. BUILD SEARCH INDEX (HYBRID) ---
# ใช้ GPU T4 สองตัวให้เป็นประโยชน์
device = "cuda" if torch.cuda.is_available() else "cpu"

print("🚀 เริ่มทำ Indexing...")
# 3.1 Sparse (BM25)
tokenized_chunks = [word_tokenize(c["text"], engine="newmm") for c in all_chunks]
bm25 = BM25Okapi(tokenized_chunks)

# 3.2 Dense (Vector) 
embed_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2", device=device)
chunk_texts = [c["text"] for c in all_chunks]
chunk_embeddings = embed_model.encode(
    chunk_texts, 
    convert_to_tensor=True, 
    normalize_embeddings=True, 
    batch_size=128, # ปรับเพิ่มได้เพราะใช้ T4
    show_progress_bar=True
)
print("✅ Indexing ทั้งหมดเสร็จสมบูรณ์!")

# --- 4. RETRIEVAL FUNCTION (พร้อมใช้) ---
def hybrid_search(query, k=5):
    # BM25
    tokenized_query = word_tokenize(query, engine="newmm")
    bm25_scores = bm25.get_scores(tokenized_query)
    
    # Vector
    query_emb = embed_model.encode([query], convert_to_tensor=True, normalize_embeddings=True)
    cosine_scores = torch.inner(chunk_embeddings, query_emb).flatten().cpu().numpy()
    
    # Normalization (ป้องกันค่าน้ำหนักเพี้ยน)
    bm25_norm = (bm25_scores - np.min(bm25_scores)) / (np.max(bm25_scores) - np.min(bm25_scores) + 1e-9)
    cosine_norm = (cosine_scores - np.min(cosine_scores)) / (np.max(cosine_scores) - np.min(cosine_scores) + 1e-9)
    
    # ผสมคะแนน (Semantic 60%, Keyword 40%)
    combined_scores = (bm25_norm * 0.4) + (cosine_norm * 0.6)
    top_indices = np.argsort(combined_scores)[::-1][:k]
    return [all_chunks[i] for i in top_indices]

KeyboardInterrupt: 